In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_percentage_error

PROJECT_ROOT = Path.cwd().resolve().parent
REPORTS_DIR = PROJECT_ROOT / "reports"


def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100

загружаю оба OOF-файла:

In [3]:
ridge_oof = pd.read_parquet(
    REPORTS_DIR / "ridge_oof_predictions_alpha_0_1.parquet"
)

catboost_oof = pd.read_parquet(
    REPORTS_DIR / "catboost_oof_predictions.parquet"
)

print("Ridge OOF:", ridge_oof.shape)
print("CatBoost OOF:", catboost_oof.shape)

assert ridge_oof["car_id"].is_unique
assert catboost_oof["car_id"].is_unique

Ridge OOF: (8340, 3)
CatBoost OOF: (8340, 4)


Склейка:

In [4]:
oof = ridge_oof.merge(
    catboost_oof[
        [
            "car_id",
            "y_true",
            "catboost_pred",
        ]
    ],
    on="car_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_ridge", "_catboost"),
)

assert len(oof) == 8340

assert np.allclose(
    oof["y_true_ridge"],
    oof["y_true_catboost"],
)

oof = oof.rename(
    columns={
        "y_true_ridge": "y_true",
    }
).drop(
    columns="y_true_catboost"
)

display(oof.head())

,car_id,y_true,ridge_pred,catboost_pred
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,40566.385555,51137.235182
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,16612.439448,13329.877806
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40958.198793,40192.222143
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,59670.249223,75309.726236
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,27712.258776,31270.977609


Теперь подберём вес CatBoost и Ridge. Начни с обычного арифметического ансамбля:

In [5]:
blend_results = []

for catboost_weight in np.arange(0, 1.01, 0.01):
    ridge_weight = 1 - catboost_weight

    blend_pred = (
        ridge_weight * oof["ridge_pred"]
        + catboost_weight * oof["catboost_pred"]
    )

    blend_results.append(
        {
            "ridge_weight": round(ridge_weight, 2),
            "catboost_weight": round(catboost_weight, 2),
            "oof_mape_pct": round(
                mape_percent(oof["y_true"], blend_pred),
                4,
            ),
        }
    )

blend_results = (
    pd.DataFrame(blend_results)
    .sort_values("oof_mape_pct")
    .reset_index(drop=True)
)

display(blend_results.head(15))

,ridge_weight,catboost_weight,oof_mape_pct
0,0.42,0.58,12.7466
1,0.41,0.59,12.7466
2,0.43,0.57,12.7478
3,0.40,0.60,12.7479
4,0.44,0.56,12.7501
5,0.39,0.61,12.7502
6,0.45,0.55,12.7531
7,0.38,0.62,12.7534
8,0.37,0.63,12.7572
9,0.46,0.54,12.7579


как по OOF выглядят сами модели без ансамбля.

In [6]:
single_model_results = pd.DataFrame(
    {
        "model": ["Ridge", "CatBoost"],
        "oof_mape_pct": [
            round(mape_percent(oof["y_true"], oof["ridge_pred"]), 4),
            round(mape_percent(oof["y_true"], oof["catboost_pred"]), 4),
        ],
    }
)

display(single_model_results)

,model,oof_mape_pct
0,Ridge,14.4690
1,CatBoost,13.5956


Мы строим ансамбль: не выбираем одну модель, а объединяем прогнозы Ridge и CatBoost.

Идея простая:

итоговый прогноз
=
ridge_weight × прогноз Ridge
+
catboost_weight × прогноз CatBoost

Например, лучший найденный вариант:

0.42 × Ridge + 0.58 × CatBoost

То есть для каждой машины мы берём:

42% прогноза Ridge;
58% прогноза CatBoost;
складываем их и получаем итоговую цену.

Зачем это нужно

Ridge и CatBoost допускают разные ошибки.

Например, по одной машине:

Истинная цена:        38 812
Ridge прогнозирует:   40 566
CatBoost прогнозирует:51 137

Здесь Ridge ближе.

По другой машине:

Истинная цена:        69 900
Ridge прогнозирует:   59 670
CatBoost прогнозирует:75 310

Почему мы используем именно OOF-прогнозы

OOF = out-of-fold predictions.

Для каждой машины прогноз получен от модели, которая не обучалась на этой машине.

Схема:

Fold 1:
обучаемся на 80% данных
предсказываем оставшиеся 20%

Fold 2:
обучаемся на другом наборе 80%
предсказываем свои 20%

...

Итог:
у каждой из 8 340 машин есть честный прогноз.

Это важно. Если бы мы смешивали прогнозы моделей на тех же объектах, на которых они обучались, ансамбль выглядел бы искусственно сильным.

Проверка ансамбля

In [7]:
ridge_holdout = pd.read_parquet(
    REPORTS_DIR / "ridge_holdout_predictions_alpha_0_1.parquet"
)

catboost_holdout = pd.read_parquet(
    REPORTS_DIR / "catboost_holdout_predictions_3000.parquet"
)

holdout = ridge_holdout.merge(
    catboost_holdout[
        [
            "car_id",
            "y_true",
            "catboost_pred",
        ]
    ],
    on="car_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_ridge", "_catboost"),
)

assert len(holdout) == 1668
assert np.allclose(
    holdout["y_true_ridge"],
    holdout["y_true_catboost"],
)

holdout = (
    holdout
    .rename(columns={"y_true_ridge": "y_true"})
    .drop(columns="y_true_catboost")
)

display(holdout.head())

,car_id,y_true,ridge_pred,catboost_pred
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,29376.245745,27571.576923
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,71057.755352,73718.393132
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,33790.401383,32367.119895
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,5180.029616,17131.541858
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,23475.926417,26695.539841


оценка фиксированного OOF-веса:

In [8]:
RIDGE_WEIGHT = 0.42
CATBOOST_WEIGHT = 0.58

holdout["blend_pred"] = (
    RIDGE_WEIGHT * holdout["ridge_pred"]
    + CATBOOST_WEIGHT * holdout["catboost_pred"]
)

holdout_comparison = pd.DataFrame(
    {
        "model": [
            "Ridge alpha=0.1",
            "CatBoost 3000",
            "Blend: 42% Ridge + 58% CatBoost",
        ],
        "holdout_mape_pct": [
            mape_percent(holdout["y_true"], holdout["ridge_pred"]),
            mape_percent(holdout["y_true"], holdout["catboost_pred"]),
            mape_percent(holdout["y_true"], holdout["blend_pred"]),
        ],
    }
).round(3)

display(holdout_comparison)

,model,holdout_mape_pct
0,Ridge alpha=0.1,14.194
1,CatBoost 3000,13.433
2,Blend: 42% Ridge + 58% CatBoost,12.493
